## Example 1: an ASGI app answers an API Gateway HTTP API (v2) / Function URL event

## Example 2: a Starlette app behind an API Gateway REST API stage (v1 event, base path stripped)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from mangum import Mangum
from starlette.applications import Starlette
from starlette.responses import PlainTextResponse
from starlette.routing import Route

# Example 2: a REST API (v1) event carries `resource` and the stage prefix in `path`; `api_gateway_base_path` strips it
# so the app's own routes match.
async def item(request):
    return PlainTextResponse(f"item {request.path_params['item_id']} q={request.query_params['q']}")

app = Starlette(routes=[Route("/items/{item_id}", item)])
event = {"resource": "/{proxy+}", "path": "/prod/items/42", "httpMethod": "GET",
         "headers": {"Host": "api.example.com", "X-Forwarded-Proto": "https"},
         "queryStringParameters": {"q": "x"}, "requestContext": {"identity": {"sourceIp": "192.0.2.1"}},
         "body": None, "isBase64Encoded": False}
with ThreadPoolExecutor(1) as pool:
    result = pool.submit(lambda: Mangum(app, lifespan="off", api_gateway_base_path="/prod")(event, None)).result(60)
print(result)
assert result["statusCode"] == 200 and result["body"] == "item 42 q=x", result
assert result["headers"]["content-type"].startswith("text/plain"), result


## Example 3: an Application Load Balancer event with multi-value headers

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from mangum import Mangum

# Example 3: an ALB target event (`requestContext.elb`) picks the ALB handler. With multi-value headers on, the
# answer carries `multiValueHeaders`, so two Set-Cookie headers both survive.
async def app(scope, receive, send):
    headers = [(b"content-type", b"text/plain"), (b"set-cookie", b"a=1"), (b"set-cookie", b"b=2")]
    await send({"type": "http.response.start", "status": 201, "headers": headers})
    await send({"type": "http.response.body", "body": b"created"})

event = {"requestContext": {"elb": {"targetGroupArn": "arn:aws:elasticloadbalancing:us-east-1:123456789012:targetgroup/t/1"}},
         "httpMethod": "POST", "path": "/things", "multiValueQueryStringParameters": {},
         "multiValueHeaders": {"host": ["lb.example.com"], "x-forwarded-for": ["192.0.2.1"]},
         "body": "", "isBase64Encoded": False}
with ThreadPoolExecutor(1) as pool:
    result = pool.submit(lambda: Mangum(app, lifespan="off")(event, None)).result(timeout=60)
print(result)
assert result["statusCode"] == 201 and result["body"] == "created", result
assert sorted(result["multiValueHeaders"]["set-cookie"]) == ["a=1", "b=2"], result


## Example 4: ASGI lifespan -- startup and shutdown run around every invocation

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from mangum import Mangum

# Example 4: with lifespan="on" Mangum runs the app's startup before the request and its shutdown after it, and the
# state the startup wrote reaches the request scope.
events = []
async def app(scope, receive, send):
    if scope["type"] == "lifespan":
        while True:
            message = await receive()
            events.append(message["type"])
            if message["type"] == "lifespan.startup":
                scope["state"]["greeting"] = "hello from startup"
            await send({"type": message["type"] + ".complete"})
            if message["type"] == "lifespan.shutdown":
                return
    await send({"type": "http.response.start", "status": 200, "headers": [(b"content-type", b"text/plain")]})
    await send({"type": "http.response.body", "body": scope["state"]["greeting"].encode()})

event = {"version": "2.0", "rawPath": "/", "headers": {}, "requestContext": {"http": {"method": "GET", "path": "/", "sourceIp": "192.0.2.1"}}}
with ThreadPoolExecutor(1) as pool:
    result = pool.submit(lambda: Mangum(app, lifespan="on")(event, None)).result(timeout=60)
print(result, events)
assert result["body"] == "hello from startup" and events == ["lifespan.startup", "lifespan.shutdown"], (result, events)


## Example 5: binary responses are base64-encoded; text types and excluded headers are configurable